In [6]:
from pathlib import Path
from pyspark.sql import SparkSession
import pandas as pd
ROOT_DIR = Path.cwd().parent
DATA_DIR = ROOT_DIR / "data"

spark = SparkSession.builder.appName("music").getOrCreate()
labeled_songs = spark.read.csv(str(DATA_DIR / "canciones" / "TRAIN_Task_1.csv"), header=True)
labeled_songs.show()

+-------------+--------------------+--------------------+-----+
|           id|          song_title|         song_artist|label|
+-------------+--------------------+--------------------+-----+
|T1_TRAIN_0001|     No Me Enseñaste|              Thalía|   NM|
|T1_TRAIN_0002|             Viajera|        Frankie Ruiz|    M|
|T1_TRAIN_0003|Te amaré toda la ...|        Javier Solís|   NM|
|T1_TRAIN_0004|      Mi lado Oscuro|Pozo Musikandante...|    M|
|T1_TRAIN_0005|    Me Enamoré de Ti|              Maluma|    M|
|T1_TRAIN_0006|     Dejame intentar|        Ulises Bueno|    M|
|T1_TRAIN_0007| El amor es libertad|     Cony la Tuquera|   NM|
|T1_TRAIN_0008|               Lento|    Daniel Santacruz|   NM|
|T1_TRAIN_0009|     La Clave de Oro|Celia Cruz y La S...|   NM|
|T1_TRAIN_0010|Amárrate a una es...|            Caifanes|   NM|
|T1_TRAIN_0011|            Mentiras|        Daniela Romo|   NM|
|T1_TRAIN_0012|        TUS LÁGRIMAS|         Mora & Sech|    M|
|T1_TRAIN_0013|             La Cita|    

In [7]:
labels = spark.read.csv(str(DATA_DIR / "canciones" / "TEST_LABELS_Task_1.csv"), header=True)
tested_songs = spark.read.csv(str(DATA_DIR / "canciones" / "TEST_Task_1.csv"), header=True)
labeled_tested_songs = tested_songs.join(
    labels,
    on="id",
    how="left" 
)

songs = labeled_songs.union(labeled_tested_songs)
print(f"df1 len: {labeled_songs.count()}, df2 len: {labeled_tested_songs.count()}")
print(f"union df len: {songs.count()}")
songs.show()

df1 len: 2104, df2 len: 527
union df len: 2631
+-------------+--------------------+--------------------+-----+
|           id|          song_title|         song_artist|label|
+-------------+--------------------+--------------------+-----+
|T1_TRAIN_0001|     No Me Enseñaste|              Thalía|   NM|
|T1_TRAIN_0002|             Viajera|        Frankie Ruiz|    M|
|T1_TRAIN_0003|Te amaré toda la ...|        Javier Solís|   NM|
|T1_TRAIN_0004|      Mi lado Oscuro|Pozo Musikandante...|    M|
|T1_TRAIN_0005|    Me Enamoré de Ti|              Maluma|    M|
|T1_TRAIN_0006|     Dejame intentar|        Ulises Bueno|    M|
|T1_TRAIN_0007| El amor es libertad|     Cony la Tuquera|   NM|
|T1_TRAIN_0008|               Lento|    Daniel Santacruz|   NM|
|T1_TRAIN_0009|     La Clave de Oro|Celia Cruz y La S...|   NM|
|T1_TRAIN_0010|Amárrate a una es...|            Caifanes|   NM|
|T1_TRAIN_0011|            Mentiras|        Daniela Romo|   NM|
|T1_TRAIN_0012|        TUS LÁGRIMAS|         Mora & Sech|

ver los generos de cada cancion para saber cual tien mas misoginia 

In [8]:
import time
from typing import Any
from datetime import datetime, timezone
from functools import cache
import requests 
import pandas as pd
from typing import Any
from pyspark.sql import functions as F, types as T
from pyspark.sql.window import Window
from pyspark.sql.utils import AnalysisException

BASE = "https://musicbrainz.org/ws/2"
HEADERS = {"User-Agent": "MisoginiaCanciones/0.1 (emilianokaleb@gmail.com)"}

def norm(col: str): 
    return F.lower(F.trim(F.regexp_replace(F.col(col), r"\s+", " ")))

songs_norm = (songs
              .withColumn("title_norm", norm("song_title"))
              .withColumn("artist_norm", norm("song_artist"))
)

pairs = (songs_norm
         .where(F.col("title_norm").isNotNull() & F.col("artist_norm").isNotNull())
         .select("title_norm", "artist_norm")
         .dropDuplicates()
)

TOTAL = pairs.count()

session = requests.Session()
session.headers.update({
    "User-Agent": "MisoginiaCanciones/0.1 (emilianokaleb@gmail.com)"
})

_last_call = 0.0

def musicbrainz_get(path:str, params:dict[str,Any], max_retries:int=5):
    global _last_call
    for intento in range(max_retries): 
        espera = 1.1 - (time.monotonic() - _last_call)
        if espera > 0: 
            time.sleep(espera)
        _last_call = time.monotonic()

        try: 
            r = session.get(f"{BASE}/{path}", 
                            params={**params, "fmt": "json"},
                            timeout=15 )
        except requests.RequestException: 
            time.sleep(2**intento)
            continue 
        if r.status_code == 503 or r.status_code >= 500: 
            time.sleep(2**intento)
            continue
        r.raise_for_status()
        return r.json()
    raise RuntimeError(f"Sin respuesta tras {max_retries} intentos: {path}") 

def lucene(s:str): 
    return s.replace("\\", "\\\\").replace('"', '\\"')

def search_recording(title: str, artist: str):
   query = f'recording:"{lucene(title)}" AND artist:"{lucene(artist)}"'
   recs = musicbrainz_get("recording", {"query": query, "limit": 1}).get("recordings", [])
   return recs[0] if recs else None

@cache  
def get_genres(entity:str, mbid:str):
   data = musicbrainz_get(f"{entity}/{mbid}", {"inc": "genres"})
   return tuple(g["name"] for g in data.get("genres", []))


def fetch_song(title:str, artist:str)->tuple[str | None, list[str], str, str | None]:
    try:
        rec = search_recording(title,artist)
        if rec is None: 
            return (None, [], "not found", None)
        genres= get_genres("recording", rec["id"])
        if not genres:  #entonces buscamos el genero del artista
            genres = get_genres("artist", rec["artist-credit"][0]["artist"]["id"])
        return (rec["id"], list(genres), "ok", None)
    except Exception as e: 
        return (None, [], "error", str(e)[:500])





In [ ]:
from pyspark.sql import DataFrame
from tqdm import tqdm
from pathlib import Path

#hacemos la carga en cache por bloques 
CACHE_PATH = DATA_DIR / "musicbrainz_cache"
BATCH_SIZE = 200

Path.mkdir(CACHE_PATH, exist_ok=True)


SCHEMA = T.StructType([ T.StructField("title_norm", T.StringType()),
    T.StructField("artist_norm", T.StringType()), 
    T.StructField("mbid", T.StringType()), 
    T.StructField("genres", T.ArrayType(T.StringType())),
    T.StructField("status", T.StringType()),
    T.StructField("error", T.StringType()),
    T.StructField("fetched_at", T.TimestampType()), 
    T.StructField("retries", T.IntegerType())
])

def read_cache()->DataFrame:
    try: 
        return spark.read.schema(SCHEMA).parquet(str(CACHE_PATH))
    except AnalysisException: 
        return spark.createDataFrame([], SCHEMA)  # pyright: ignore[reportUnknownMemberType]


def pending(retry_errors:bool): 
    facts = read_cache()
    if retry_errors:
        facts = facts.where(F.col("status") != "error")
    return (pairs
            .join(
                facts.select("title_norm", "artist_norm"), 
                on=["title_norm", "artist_norm"], 
                how="left_anti")
        
        )
    
def process(retry_errors:bool=True):
    with tqdm(total = TOTAL, unit='canciones', unit_scale=True, desc="Recuperando generos") as pbar:
        while True: 
            batch = pending(retry_errors).limit(BATCH_SIZE).collect()
            if not batch: 
                break
            rows = [
                (
                    r.title_norm, 
                    r.artist_norm, 
                    *fetch_song(r.title_norm, r.artist_norm),
                    datetime.now(timezone.utc)
                )
                for r in batch
            ]
            #es un dataframe temporal con los datos del batch
            spark.createDataFrame(rows, SCHEMA).write.mode("append").parquet(str(CACHE_PATH))# pyright: ignore[reportUnknownMemberType]
            pbar.update(len(rows))
            


In [10]:
#ahora usaremos todos lo metodos para poder consumir adecuadamente
process()

Recuperando generos:   0%|          | 0.00/2.63k [00:00<?, ?canciones/s]


AttributeError: 'NoneType' object has no attribute 'where'

In [ ]:
w = Window.partitionBy("title_norm", "artist_norm").orderBy(F.col("fetched_at").desc())
genres = (read_cache()
          .withColumn("rn", F.row_number().over(w))
          .where(F.col("rn") == 1)
          .drop("rn", "error")
)
songs_with_genre = songs_norm.join(genres, on=["title_norm", "artist_norm"], how="left")
songs_with_genre.groupBy("status").count().show()